<a href="https://colab.research.google.com/github/pablorubal/tfg-embeddings/blob/main/01_visualization_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap

In [ ]:
#Triger authentication flow
ee.Authenticate()

# Initialize the library
ee.Initialize(project='gen-lang-client-0148391940')

In [ ]:
!pip install osmnx

In [ ]:
import osmnx as ox

# Obtiene la geometría de Burela
gdf = ox.geocode_to_gdf("Burela, Lugo, Spain")
# Extraer el MultiPolygon
polygon = gdf.geometry.iloc[0]
coords = list(polygon.exterior.coords)
geometry = ee.Geometry.Polygon(coords)

In [ ]:
# geoJSON = {
#   "type": "FeatureCollection",
#   "features": [
#     {
#       "type": "Feature",
#       "properties": {},
#       "geometry": {
#         "coordinates": [
#           [
#             [
#               -7.408043908593299,
#               43.45464557006821
#             ],
#             [
#               -7.408043908593299,
#               43.413658496069786
#             ],
#             [
#               -7.328783243763866,
#               43.413658496069786
#             ],
#             [
#               -7.328783243763866,
#               43.45464557006821
#             ],
#             [
#               -7.408043908593299,
#               43.45464557006821
#             ]
#           ]
#         ],
#         "type": "Polygon"
#       }
#     }
#   ]
# }

# coords = geoJSON["features"][0]["geometry"]["coordinates"]
# geometry = ee.Geometry.Polygon(coords)


In [ ]:
mapa = geemap.Map()
mapa.setOptions('SATELLITE')
mapa.centerObject(geometry, 14)
mapa.addLayer(geometry, {'color':'red'}, 'geometriaBurela')

In [ ]:
embeddings = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
startDate = ee.Date.fromYMD(2024, 1, 1)
endDate = startDate.advance(1, 'year')

filter_embeddings = embeddings.filter(ee.Filter.date(startDate, endDate)).filter(ee.Filter.bounds(geometry))

# embeddings_mosaic = filter_embeddings.mosaic() # combina todas las imágenes de la colección en una sola, en el caso del municipio solo hay una, por lo que podríamos
                                               # usar first() para sacar la imagen de la colección, pero así ya queda para cuando sean más.
embeddings_median = filter_embeddings.median()

In [ ]:
print(filter_embeddings.size().getInfo())
print(embeddings_mosaic.getInfo())

In [ ]:
print(embeddings_median.getInfo())

In [ ]:
vis_params = {
    'min': -0.3,
    'max': 0.3,
    'bands': ['A11', 'A22', 'A33'] #podemos poner las 3 bandas que queramos
}

In [ ]:
mapa.addLayer(embeddings_median.clip(geometry), vis_params, '3 bandas de embeddings')

# Unsupervised clustering

In [ ]:
# extraemos puntos para el entrenamiento

nSamples = 1000
training = embeddings_median.sample(
    region = geometry,    # que sean puntos dentro de burela
    scale = 10,           # resolución nominal en metros
    numPixels = nSamples,     # número total de samples/pixels
    seed = 100,        # semilla para reproducibilidad
    geometries = True,     #
    tileScale = 8     # para subidivir la tarea
)
print(training.first().getInfo()) # para ver los valores del primer punto

In [ ]:
def fixedClusters(nClusters, embeddings_image = embeddings_median, training_data = training):
  clusterer = ee.Clusterer.wekaKMeans(nClusters=nClusters).train(training_data)
  clustered = embeddings_image.cluster(clusterer)
  mapa.addLayer(clustered.randomVisualizer().clip(geometry), {}, f'{nClusters} Fixed clusters')

In [ ]:
fixedClusters(3)

In [ ]:
def CascadeClusters( minClusters, maxClusters, embeddings_image = embeddings_median, training_data = training):
  clusterer = ee.Clusterer.wekaCascadeKMeans(minClusters=minClusters, maxClusters=maxClusters).train(training_data)
  clustered = embeddings_image.cluster(clusterer)
  mapa.addLayer(clustered.randomVisualizer().clip(geometry), {}, f'Cascade Clustering ({minClusters} to {maxClusters})')

In [ ]:
CascadeClusters(4, 7)

In [ ]:
def XmeansCluster(minClusters, maxClusters, maxIterations=3, embeddings_image = embeddings_median, training_data = training):
  clusterer = ee.Clusterer.wekaXMeans(minClusters=minClusters, maxClusters=maxClusters, distanceFunction="cosin").train(training_data)
  clustered = embeddings_image.cluster(clusterer)
  mapa.addLayer(clustered.randomVisualizer().clip(geometry), {}, f'XMeans Clustering ({minClusters} to {maxClusters} with {maxIterations} max iterations)')

In [ ]:
XmeansCluster(4, 7)

### Función para clasificación no supervisada usando el coseno como función de distancia

Hay que hacer primero normalización L2 de los datos

In [ ]:
# l2_norm = embeddings_mosaic.pow(2).reduce(ee.Reducer.sum()).sqrt()
l2_norm = embeddings_median.pow(2).reduce(ee.Reducer.sum()).sqrt()

# embeddings_norm = embeddings_mosaic.divide(l2_norm)
embeddings_norm = embeddings_median.divide(l2_norm)

nSamples = 1000
training_norm = embeddings_norm.sample(
    region = geometry,
    scale = 10,
    numPixels = nSamples,
    seed = 100,
    geometries = True,
    tileScale = 8
)

print(training_norm.first().getInfo())

In [ ]:
print(training_norm.first().getInfo())

In [ ]:
estadisticas_l2 = l2_norm.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=geometry,
    scale=10,          # La misma resolución que usaste en el sample
    maxPixels=1e9      # Límite de seguridad por si la zona es muy grande
).getInfo()

print("Estadísticas de la norma L2 de los embeddings originales:")
print(estadisticas_l2)

#### Embeddings ya normalizados!
Creo que no es necesario esto, así que podemos asumir que al usar la distancia euclidea en este caso, se está intentando minimizar esta, lo que resulta en maximizar la similaridad coseno

In [ ]:
fixedClusters(10)

## Clustering jerárquico
En ee no he encontrado algo con lo que hacer cluster jerárquico, así que uso sklearn, por lo que hay que convertir los datos a pandas

In [ ]:
import pandas as pd
import numpy as np

datos = training.getInfo()

propiedades = [feature['properties'] for feature in datos['features']]

df = pd.DataFrame(propiedades)

print(df.shape)

In [ ]:
df.head()

In [ ]:
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as shc

plt.figure(figsize=(10, 7))
plt.title("Dendrograma de Embeddings en Burela")

# 'ward' minimiza la varianza dentro de los clústeres, como los datos ya están normalizados en L2 se supone que funciona bien
dendrograma = shc.dendrogram(shc.linkage(df, method='ward'))

plt.xlabel("Píxeles de muestra")
plt.ylabel("Distancia")
plt.show()

### Con el dendograma podemos hacernos una idea de cuántos clusters usar, pero para estar seguro uso métricas matemáticas (silueta, calinski y davies)

In [ ]:
# viendo el dendograma el número óptimo de clusters estará entre 2 y 5
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import davies_bouldin_score, silhouette_score, calinski_harabasz_score

for k in range(2,6):
  clusterer = AgglomerativeClustering(n_clusters=k)  # por defecto ya usa distancia euclídea y ward
  clustered = clusterer.fit_predict(df)

  davies = davies_bouldin_score(df, clustered)
  silueta = silhouette_score(df, clustered)
  calinski = calinski_harabasz_score(df, clustered)

  print(f"Evaluando para {k} clusters:")
  print(f"Davies (más bajo mejor): {davies}")
  print(f"Silueta (más alto mejor, va de -1 a 1): {silueta}")
  print(f"Calinski (más alto mejor): {calinski}\n")


### Viendo los resultados vemos que el mejor número de clusters es 2, al tener menor score de Davies y mayor Silueta y Calinski (Silueta suele ser el más robusto)

### Ahora vamos a hacer clasificación semisupervisada: le paso las etiquetas generadas antes (no supervisadas) a un clasificador supervisado (como knn,  random forest o xgboost)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=10, metric="cosine")
neigh.fit(df, clustered)


In [ ]:
print(embeddings_median.getInfo())

In [ ]:
puntos_total = embeddings_median.sample(
    region = geometry,
    scale = 10,
    geometries = True,
    tileScale = 8
)


datos_total = puntos_total.getInfo()

propiedades_total = [feature['properties'] for feature in datos_total['features']]

df_total = pd.DataFrame(propiedades_total)

# Add satellite imagery

In [ ]:
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(geometry)
    .filterDate('2024-01-01', '2024-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

image = collection.median().clip(geometry) # .clip() is optional if you only want the exact shape

# 2. Define Visualization Parameters
# We select the Red (B4), Green (B3), and Blue (B2) bands for a true-color image
vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.4 # Gamma correction to make the image look more natural
}

# 3. Add the Layers
# Add the satellite image first
mapa.addLayer(image, vis_params, 'Sentinel-2 True Color')

In [ ]:
display(mapa)